# Interactive eigenstate/overlap session

`Calculation.eigenstate_session()` (task 9002, elkpy Fortran extension --
`patches/0003-eigenstate-session.patch`, `src/elkpy_eigenstates.f90`) starts a
long-lived `elk` subprocess that answers repeated eigenstate/overlap queries at
arbitrary k-points over a stdin/stdout protocol, avoiding paying Elk's ground-
state-dependent setup cost (`readstate`, `genvsig`, `linengy`, `genapwlofr`,
`gensocfr`) again for every single query. See `docs/design.md` #14 and
`docs/physics.tex` Part III for why `evecsv` (not `evecfv`) is a valid raw-overlap
basis, and why cross-k overlaps need the `genwfsvp`/`genolpq` route.

`get_eigenstates()`/`get_overlap()` are thin one-off wrappers that open and close
their own session -- prefer `eigenstate_session()` directly (as below) whenever
making more than one query, to actually get the "stay warm" benefit.

In [1]:
import numpy as np

from elkpy.structure import Structure

SI_AVEC = [(5.13, 5.13, 0.00), (5.13, 0.00, 5.13), (0.00, 5.13, 5.13)]
SI_SPECIES = {"Si": [(0.0, 0.0, 0.0), (0.25, 0.25, 0.25)]}

calc = Structure(SI_AVEC, SI_SPECIES).get_calculation(
    "_scratch/si_eigenstates", xc="PW", ngridk=(4, 4, 4)
)
calc.get_energy()  # warm the cached ground state

-578.07279182424

## `get_eigenstates()`: energies and `evecsv` at one k-point

Second-variational energies and eigenvectors via fresh on-the-fly diagonalisation
-- no requirement that `k` sit on the ground state's own `ngridk` mesh. `evecsv` is
built from an already-orthonormalized first-variational basis, so its Gram matrix
must be the identity to ordinary numerical precision, independent of any reference
data -- a good sanity check with no external comparison needed.

In [2]:
state = calc.get_eigenstates((0.1, 0.2, 0.05))
print("energies (Hartree):", state.energies[:6])
print("evecsv shape:", state.evecsv.shape)

gram = state.evecsv.conj().T @ state.evecsv
print("max |evecsv^H evecsv - I|:", np.abs(gram - np.eye(gram.shape[0])).max())

energies (Hartree): [-0.22875843  0.11718175  0.15723857  0.18319185  0.28835055  0.30931346]
evecsv shape: (13, 13)
max |evecsv^H evecsv - I|: 0.0


## `eigenstate_session()`: many queries, one warm process

Used as a context manager. Below: the same k-point queried twice through one
session -- the results must agree to essentially machine precision, since nothing
about the underlying diagonalisation changes between the two calls.

In [3]:
kpoints = [(0.0, 0.0, 0.0), (0.1, 0.0, 0.0), (0.1, 0.1, 0.0), (0.1, 0.1, 0.1)]

with calc.eigenstate_session() as session:
    first_pass = [session.get_eigenstates(k).energies[:4] for k in kpoints]
    second_pass = [session.get_eigenstates(k).energies[:4] for k in kpoints]

max_diff = max(np.abs(np.array(a) - np.array(b)).max() for a, b in zip(first_pass, second_pass))
print("max energy difference between two passes over the same session:", max_diff)

max energy difference between two passes over the same session: 0.0


## `get_overlap()` / `session.overlap()`: comparing eigenstates across k-points

$\langle \psi_a(k_a) | \psi_b(k_b) \rangle$ for a contiguous band window -- the only
valid way to compare eigenstates from two different diagonalisations (see
`docs/physics.tex` Part III for why). Overlapping a k-point with itself must give
the identity matrix; the tolerance is looser than machine precision because
`genwfsvp`/`genolpq`'s muffin-tin-plus-interstitial real-space expansion has its
own inherent truncation error (angular momentum cutoff, interstitial G-vector
cutoff) at ordinary `rgkmax` -- observed at the ~1e-3 level even for a single
non-degenerate band.

In [4]:
with calc.eigenstate_session() as session:
    self_overlap = session.overlap((0.1, 0.2, 0.05), (0.1, 0.2, 0.05), ist0=1, ist1=4)

print("<psi(k)|psi(k)>:")
print(np.round(self_overlap, 4))
print("max |self_overlap - I|:", np.abs(self_overlap - np.eye(4)).max())

<psi(k)|psi(k)>:
[[ 9.9990e-01-0.j      5.0000e-04-0.0004j  4.0000e-04+0.0002j
   1.0000e-04+0.0002j]
 [ 5.0000e-04+0.0004j  1.0008e+00+0.j      3.0000e-04+0.0006j
  -2.0000e-04-0.0002j]
 [ 4.0000e-04-0.0002j  3.0000e-04-0.0006j  9.9990e-01-0.j
  -4.0000e-04+0.001j ]
 [ 1.0000e-04-0.0002j -2.0000e-04+0.0002j -4.0000e-04-0.001j
   9.9900e-01-0.j    ]]
max |self_overlap - I|: 0.0011178384260165945


And an overlap between two genuinely different k-points -- band 1 (non-degenerate
at Gamma, unlike bands 2-4) between Gamma and its `ngridk=(2,2,2)` mesh neighbour in
direction 1:

In [5]:
cross_calc = Structure(SI_AVEC, SI_SPECIES).get_calculation(
    "_scratch/si_eigenstates_mesh", xc="PW", ngridk=(2, 2, 2)
)
cross_overlap = cross_calc.get_overlap((0.0, 0.0, 0.0), (0.5, 0.0, 0.0), ist0=1, ist1=1)
print("<psi_1(Gamma)|psi_1(Gamma+e1)>:", cross_overlap[0, 0])

<psi_1(Gamma)|psi_1(Gamma+e1)>: (0.7694394123928473-0.0001454508279447272j)


## Recap

Six notebooks, six feature areas:

1. `01_getting_started.ipynb` -- `Structure`/`Calculation`, energy, bands, DOS.
2. `02_relaxation_forces_and_properties.ipynb` -- forces, relaxation, effective
   mass, density, `run_tasks()`.
3. `03_phonon_dispersion_and_dos.ipynb` -- DFPT phonons.
4. `04_per_species_soc_scaling.ipynb` -- per-species SOC scaling.
5. `05_berry_curvature.ipynb` -- Berry curvature / Chern numbers, mesh and path
   modes, h-BN K/K' valleys.
6. `06_eigenstate_session.ipynb` -- this notebook.

See `README.md` for the project overview and `docs/design.md`/`docs/physics.tex`
for the full architecture and physics writeups behind each of these.